In [0]:
%sql
USE CATALOG ws_banking_etl2;
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
transactions = spark.read.table("bronze.transactions_data")
cards = spark.read.table("bronze.cards_data")
users = spark.read.table("bronze.users_data")
mcc = spark.read.table("bronze.mcc_codes")
fraud = spark.read.table("bronze.train_fraud_labels")


In [0]:
from pyspark.sql.functions import col, when

# Convert fraud labels to boolean and cast ID to integer
fraud_clean = (
    fraud
        .withColumn("transaction_id", col("id").cast("int"))
        .withColumn(
            "is_fraud",
            when(col("target") == "Yes", True).otherwise(False)
        )
        .select("transaction_id", "is_fraud")
)


In [0]:
from pyspark.sql.functions import col

#Cast mcc_code to short to match transactions.mcc
mcc_clean = mcc.withColumn(
    "mcc",
    col("mcc_code").cast("short")
).select(
    "mcc",
    col("description").alias("mcc_description")
)


In [0]:
from pyspark.sql.functions import col

# Rename transaction id first for clarity
transactions_clean = transactions.withColumnRenamed("id", "transaction_id")

# Join fraud + MCC
transactions_enriched = (
    transactions_clean
        .join(fraud_clean, on="transaction_id", how="left")
        .join(mcc_clean, on="mcc", how="left")
)


In [0]:
# no need for client_id col as we have it in transactions, so just kepp the necessary cols
cards_clean = (
    cards
        .withColumnRenamed("id", "card_id")
        .select(
            "card_id",
            "card_brand",
            "card_type",
            "credit_limit",
            "has_chip"
        )
)

users_clean = (
    users
        .withColumnRenamed("id", "user_id")
        .select(
            "user_id",
            "current_age",
            "gender",
            "credit_score",
            "yearly_income"
        )
)


transactions_enriched = (
    transactions_enriched
        .join(cards_clean, on="card_id", how="left")
        .join(users_clean, transactions_enriched.client_id == users_clean.user_id, how="left")
)


In [0]:
from pyspark.sql.functions import to_date

# Ensure date column name is clean
transactions_enriched = transactions_enriched.withColumn(
    "transaction_date",
    to_date("date")
).drop("date")

transactions_enriched = transactions_enriched.fillna({"is_fraud": False})

transactions_enriched = transactions_enriched.drop("user_id")


In [0]:
transactions_enriched.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.transactions")

users_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.users")

cards_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.cards")


In [0]:
transactions_enriched.printSchema()


root
 |-- card_id: short (nullable = true)
 |-- mcc: short (nullable = true)
 |-- transaction_id: integer (nullable = true)
 |-- client_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- errors: string (nullable = true)
 |-- is_fraud: boolean (nullable = false)
 |-- mcc_description: string (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- yearly_income: string (nullable = true)
 |-- transaction_date: date (nullable = true)



In [0]:
silver = spark.read.table("silver.transactions")

silver.select("is_fraud").groupBy("is_fraud").count().show()


+--------+--------+
|is_fraud|   count|
+--------+--------+
|   false|13292583|
|    true|   13332|
+--------+--------+

